In [170]:
from src.utils import setup_client
import requests
import numpy as np
import pandas as pd
import re
import math

In [171]:
hf_client = setup_client()

✅ Connected to TUDelft LLM proxy


In [172]:
def query_model(prompt, model_name, max_tokens, client):
    """
    Parameters
    ----------
    prompt     : str   — the input text
    model_name : str   — "gpt2", "llama-1b", or "llama-8b"
    max_tokens : int   — how many tokens to generate (capped at 200 by proxy)
    client     : str   — the proxy URL returned by setup_client()

    Returns
    -------
    dict with keys:
        "answer"      : str   — the generated text
        "logprobs"    : dict  — {token: logprob}
        "token_probs" : dict  — {token: probability}
    """
    response = requests.post(
        f"{client}/generate",
        json={
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )

    if response.status_code == 429:
        print("⏳ Rate limit hit — wait a moment and try again.")
        return None

    if response.status_code == 400:
        print(f"❌ Bad request: {response.json().get('detail')}")
        return None

    response.raise_for_status()
    return response.json()

# Semantic entropy exercise
In this notebook, we will color code answers of the LLM based on the *semantic entropy* of the answer. You have all read the blogpost on the semantic entropy method (https://oatml.cs.ox.ac.uk/blog/2024/06/19/detecting_hallucinations_2024.html), so we assume you know the general idea of the method. In order to color code the answers of the LLM, we will slightly alter the semantic entropy pipeline by decomposing the answer to the main question into *sentence parts* and extracting one or more factoids from each sentence part. This makes it easier to color code the answer in the end, we can give each sentence part a color based on the average semanatic confidence of all the factoids that were extracted from this sentence part. An overview of the pipeline we will follow in this exercise can be seen below:

1. Generate one long answer
2. Decompose answer into *sentence parts*
3. Extract one or more factoids from each sentence part
4. Use LLM to turn each factoid into Q questions
5. Use LLM to answer each question M times
6. For each question, compute the semantic entropy over the M answers, including the original factual claim
7. Compute average semantic entropy over Q questions
8. Use this average as the semantic entropy over the factoid
9. Calculate semantic confidence of sentence part based on semantic entropy of the extracted factoids
10. Color code the sentence part based on this semantic confidence


The goal of this exercise is to get an idea of how the semantic entropy method works, what its advantages and disadvantages are and to computationally implement part of the method. We will guide you through all the steps of the pipeline. For most steps the code will be provided, you can just run it and analyze what is happening. For steps **[insert which steps]** you need to write some code yourself.

### Step 1 - Generate long answer

In [177]:
# initialize global variables
model_name = "llama-8b"
max_tokens = 1000
client = hf_client

In [165]:
def generate_long_answer(question, model_name, max_tokens, client):
    '''
    Generates a longer answer to the provided main question. 
    Returns a string that contains the answer.
    '''

    prompt = question + '\nAnswer in exactly 3 sentences. Do not use bullet points in your answer.'
    response = query_model(prompt, model_name, max_tokens, client)
    answer = response['answer']
    return answer


Try out the function by running the cell below. **You can change the question to a question of your interest.** It helps if you know a lot about the topic of the question and the LLM most likely will not. This way, you can easily spot confabulations in the answer yourself and later verify those by calculating the semantic confidence.

In [174]:
question = "Can you give me examples of student rowing associations in Delft?" # TO DO: change to something of your interest
answer = generate_long_answer(question, model_name, max_tokens, client)
print(answer)

One example of a student rowing association in Delft is the "Willem III Studenten Roeivereniging", which is the oldest rowing association in the Netherlands established in 1876. The association "ASR" or "Artemis Studenten Roeivereniging" is another student rowing association in Delft, providing opportunities for rowing and boat racing to the students. "Willem III SR" has a strong history in competition at both national and international levels which has created a high level of rowing excellence in Delft.


### Step 2 & 3 - Decompose answer into sentence parts and extract factoids for each sentence part

## Original function for decomposition:

In [178]:
def decompose_into_sentence_parts_and_factoids(answer, model_name, max_tokens, client):
    '''
    Decomposes the long answer given by the model into 'sentence parts', which is a part of a sentence in the answer 
    that contains one or more factual propositions. 
    Args:
        - answer: the long answer generated by the model
        - model_name: the name of the LLM model that is used
        - max_tokens: the max number of output tokens the model is allowed to give
    Returns:
        - s_part_to_factoid: dictionary where the key is a sentence part and the value a factoid. dict['sentence_part'] = 'factoid'
        - factoid_to_s_part: dictionary where the key is a factoid and the sentence part the value. dict['factoid'] = 'sentence_part'
        - sentence_parts: list of all the sentence parts
        - factoids: list of all the factoids
    '''
    full_prompt = answer + "\n \n Please split the answer above into sentences. Then split the sentences into different 'sentence parts'. "
    full_prompt += "The sentence parts must be unique and exactly correspond to one part of the original answer. "
    full_prompt += "A sentence part can not include parts that were already part of a previous sentence part."
    full_prompt += "Each 'sentence part' should contain at least one verb. " # added this to avoid meaningless sentence parts
    full_prompt += "For each sentence part, list the factoids that can be derived from that sentence part, if possible."
    full_prompt += "Please be as specific as possible in the factoid that you generate, "
    full_prompt += "e.g. always provide the names of the object/topic/person/occurrence in the factoid that you generate. "
    full_prompt += "Be complete and do not leave any factoids out. Each sentence part can have multiple factoids that were derived from it. \n"
    full_prompt += "However, you must list them seperately. "
    full_prompt += "Formulate your answer like this: \n"
    full_prompt += "*<first sentence part> : <factual proposition1> \n*<first sentence part> : <factual proposition2>\n" 
    full_prompt += "*<second sentence part> : <factual proposition1> \n*<second sentence part> : ..."
    full_prompt += "You must only answer with the sentence parts and the factual propositions, nothing else. "
    
    # sending prompt, asking the model to split the answer into factoids
    try:
        response = query_model(full_prompt, model_name, max_tokens, client)
    
    except Exception as e:
        print(f"Error during API call: {e}. Skipping function call.")

    # initializing return datastructures
    sentence_part_to_factoid = {}
    factoid_to_sentence_part = {}
    sentence_parts = []
    factoids = []
    
    # split the response into sentence parts and factoids
    print(f'ORIGINAL ANSWER: {answer}')
    print('-'*100)
    print(f'ANSWER AFTER SPLITTING INTO SENTENCE PART AND FACTOIDS:\n')
    print(response['answer'])
    print('-'*100)
    response_split = response['answer'].split('*')
    print(f'ANSWER AFTER SPLITTING ON * \n')
    print(response_split)
    print('-'*100)
    for i, bullet_point in enumerate(response_split):
        if i == 0:
            continue
        split_bullet_point = bullet_point.split(':')
        print(f'ANSWER AFTER SPLITTING ON : \n')
        print(split_bullet_point)

        sentence_part = split_bullet_point[0].strip()
        factoid = split_bullet_point[1].strip()

        sentence_part_to_factoid[sentence_part] = factoid
        factoid_to_sentence_part[factoid] = sentence_part

        sentence_parts.append(sentence_part)
        factoids.append(factoid)
        
    return sentence_part_to_factoid, factoid_to_sentence_part, sentence_parts, factoids
    

In [179]:
sentence_part_to_factoid, factoid_to_sentence_part, sentence_parts, factoids = decompose_into_sentence_parts_and_factoids(answer, model_name, max_tokens, client)

print("\nDECOMPOSITION INTO SENTENCE PARTS AND FACTOIDS")
print("-"*80)
print(f"Number of factoids: {len(factoids)}. Number of sentence parts: {len(sentence_parts)}. ")
for i, (factoid, sentence_part) in enumerate(factoid_to_sentence_part.items(), 1):
    print(f"[{i}] Sentence Part:")
    print(f"    {sentence_part}")
    print(f"    → Factoid:")
    print(f"      {factoid}\n")

ORIGINAL ANSWER: One example of a student rowing association in Delft is the "Willem III Studenten Roeivereniging", which is the oldest rowing association in the Netherlands established in 1876. The association "ASR" or "Artemis Studenten Roeivereniging" is another student rowing association in Delft, providing opportunities for rowing and boat racing to the students. "Willem III SR" has a strong history in competition at both national and international levels which has created a high level of rowing excellence in Delft.
----------------------------------------------------------------------------------------------------
ANSWER AFTER SPLITTING INTO SENTENCE PART AND FACTOIDS:

One example of a student rowing association in Delft is the "Willem III Studenten Roeivereniging" 
*one example exists *: factoid related to a student rowing association in Delft exists * 
*a student rowing association exists *: factoid related to a student rowing association exists * 

the "Willem III Studenten R

IndexError: list index out of range

# New propoposed functions using system prompt
This includes:
1. query_model(...) function that now also takes system_prompt as an input
2. extract_raw_json_pipeline(...) that makes 4 API calls in total to extract sentence parts and factoids from answer. Outputs raw JSON
3. parse_json_to_dictionaries(...) parses the JSON and puts data into designated dictionaries


In [ ]:
def query_model(system_prompt, prompt, model_name, max_tokens, client):
    """
    Parameters
    ----------
    system_prompt : str   — the system prompt
    prompt        : str   — the input text
    model_name    : str   — "gpt2", "llama-1b", or "llama-8b"
    max_tokens    : int   — how many tokens to generate (capped at 200 by proxy)
    client        : str   — the proxy URL returned by setup_client()

    Returns
    -------
    dict with keys:
        "answer"      : str   — the generated text
        "logprobs"    : dict  — {token: logprob}
        "token_probs" : dict  — {token: probability}
    """
    
    # Also include SYSTEM PROMPT!
    response = requests.post(
        f"{client}/generate",
        json={
            "system_prompt": system_prompt, 
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )
    
    if response.status_code == 429:
        print("⏳ Rate limit hit — wait a moment and try again.")
        return None

    if response.status_code == 400:
        print(f"❌ Bad request: {response.json().get('detail')}")
        return None

    response.raise_for_status()
    return response.json()

In [ ]:
def extract_raw_json_pipeline(answer, model_name, max_tokens, client):
    """
    Makes 1 API call to split the answer into sentences, then 1 API call per sentence 
    to extract sentence parts and factoids
    Returns raw parsed JSON
    """
    print("=" * 80)
    print("PHASE 1: SENTENCE SPLITTING (JSON)")
    print("=" * 80)
    
    sys_prompt_1 = "You are a strict text parser. You ONLY output valid JSON arrays of strings. No markdown, no explanations."
    user_prompt_1 = f"""Split the following text into individual sentences.
    Rules:
    1. Output a maximum of 3 sentences.
    2. Respond ONLY with a valid JSON array of strings.
    
    Text: "{answer}"
    
    Example Output:
    ["Sentence 1.", "Sentence 2.", "Sentence 3."]
    """
    
    # First API call: split the sentences using LLM
    try:
        res_sentences = query_model(sys_prompt_1, user_prompt_1, model_name, max_tokens, client)
        raw_sentences = res_sentences.get('answer', '').strip()
        clean_json_str = re.sub(r'```json|```', '', raw_sentences).strip() # safety cleaner to process potential "```json" in answer
        sentences = json.loads(clean_json_str)
        print(f"Successfully extracted {len(sentences)} sentences via JSON.")
    except Exception as e:
        print(f"❌ Error during Phase 1: {e}")
        return [], []

    print("\n" + "=" * 80)
    print("PHASE 2: FACTOID EXTRACTION (JSON)")
    print("=" * 80)
    
    master_json_output = []
    sys_prompt_2 = "You are a strict data extraction tool. You ONLY output valid JSON arrays of objects. You NEVER invent external information."
    
    # API calls 2-4: split into 2 sentence parts, and split those into 2 factoids
    for i, sentence in enumerate(sentences, 1):
        print(f"Processing Sentence {i}/{len(sentences)}: '{sentence}'")
        
        user_prompt_2 = f"""Decompose the given sentence into consecutive 'sentence parts' and extract factoids.
                
        RULES:
        1. Min 1, max 2 sentence parts total.
        2. EXACT COPY: The 'sentence part' MUST be an exact, unaltered substring copied directly from the original sentence.
        3. Min 1, max 2 brief factoids per sentence part.
        4. STRICT: ONLY extract facts explicitly stated.
        5. Respond ONLY with a valid JSON array.
        6. Be exact in the factoids that you generate: provide names of the people/places/occurences that you mention.
        
        EXAMPLE INPUT:
        Sentence: "Marie Curie was born in Warsaw, and she later moved to Paris to study."
        
        EXAMPLE OUTPUT:
        [
          {{
            "sentence_part": "Marie Curie was born in Warsaw,",
            "factoids": ["Marie Curie was born in Warsaw"]
          }},
          {{
            "sentence_part": "and she later moved to Paris to study.",
            "factoids": ["Marie Curie moved to Paris", "Marie Curie moved to Paris for her studies"]
          }}
        ]
        
        ACTUAL INPUT:
        Sentence: "{sentence}"
        """
        
        try:
            res_factoids = query_model(sys_prompt_2, user_prompt_2, model_name, max_tokens, client)
            raw_output = res_factoids.get('answer', '').strip()
            clean_output = re.sub(r'```json|```', '', raw_output).strip() # again safety check to remove additional json stuff
            
            # Parse the JSON and append to master list
            parsed_sentence_data = json.loads(clean_output)
            master_json_output.append(parsed_sentence_data)
            
        except Exception as e:
            print(f"❌ Error parsing JSON for sentence {i}: {e}")
            master_json_output.append([]) # Append empty list on failure to keep index aligned
            
    return master_json_output, sentences

In [183]:
def parse_json_to_dictionaries(master_json_output, original_sentences):
    """
    Takes the raw JSON output from the LLM pipeline, applies Python code with safety limits,
    and puts it into the designated dictionaries
    """
    # define the output dictionaries
    sentence_part_to_factoids = {}
    factoid_to_sentence_part = {}
    sentence_parts = []
    factoids_list = []
    
    for sentence_data, original_sentence in zip(master_json_output, original_sentences):
        
        # Strictly limit to max 2 sentence parts per sentence
        constrained_parts = sentence_data[:2] 
        
        for item in constrained_parts:
            s_part = item.get("sentence_part", "").strip()
            extracted_factoids = item.get("factoids", [])
            
            if not s_part:
                continue
                
            # check if sentence part exactly matches part of the original sentence
            if s_part not in original_sentence:
                print(f"WARNING: Paraphrase detected! \nExpected substring of: '{original_sentence}' \nGot: '{s_part}'")
            
            # Strictly limit to max 2 factoids per part
            constrained_factoids = [f.strip() for f in extracted_factoids if f.strip()][:2]

            if not constrained_factoids:
                continue
                
            # Fill our data structures
            sentence_parts.append(s_part)
            sentence_part_to_factoids[s_part] = constrained_factoids
            
            for factoid in constrained_factoids:
                factoids_list.append(factoid)
                factoid_to_sentence_part[factoid] = s_part

    return sentence_part_to_factoids, factoid_to_sentence_part, sentence_parts, factoids_list

In [ ]:
# Run the API calls
raw_json_data, original_sentences = extract_raw_json_pipeline(answer, model_name, max_tokens, client)

# Parse the json output
s_part_to_factoids, factoid_to_s_part, sentence_parts, factoids = parse_json_to_dictionaries(raw_json_data, original_sentences)

print("\n" + "=" * 80)
print("FINAL PARSED DATA STRUCTURE")
print("=" * 80)
for i, s_part in enumerate(sentence_parts, 1):
    print(f"[{i}] '{s_part}'")
    for factoid in s_part_to_factoids[s_part]:
        print(f"    ↳ {factoid}")